# <center>Pronóstico Parados / Afiliados / Demandantes — TimesFM 2.5 (recursivo)</center>
## <center>Variante de `forecast_ABC_estatal_TimesFM2.5_mejorado.ipynb` — pronóstico encadenado en bloques de 3 meses</center>

> Este notebook es una variante del notebook principal (`forecast_ABC_estatal_TimesFM2.5_mejorado.ipynb`),
> reutilizando todo lo ya verificado (carga del modelo, SSL/proxy, funciones de fine-tuning, `force_flip_invariance=False`,
> escala real sin logaritmo). La única diferencia real es **cómo se genera el pronóstico final**:
> en vez de una única llamada a `model.forecast(horizon=36-40, ...)`, se encadenan varias llamadas de
> `RECURSIVE_STEP=3` meses cada una, añadiendo lo predicho en cada paso al contexto del siguiente —
> igual que hace `decode()` internamente cuando el horizonte pedido supera su bloque nativo de
> `output_patch_len=128` (aquí, en vez de bloques de 128, se usan bloques de 3).

**Simplificaciones respecto al notebook principal, a petición expresa:**
- **Sin CV / grid search**: no se recorren combinaciones de hiperparámetros ni folds — el fine-tuning
  se hace una sola vez con los valores ya conocidos por la CV del notebook principal:
  `layers=4`, `lr=1e-5`.
- El **Zero-Shot no se ve afectado** por nada de esto — es el modelo preentrenado tal cual, sin
  fine-tuning; no depende de `layers` ni `lr`.
- **Histórico completo** (desde el inicio del CSV hasta el último dato disponible) como contexto,
  horizonte dinámico hasta diciembre del año actual + 3 — igual que el notebook principal.
- Tanto el **backtest** (últimos `VAL_MONTHS` meses, para tener un MAPE comparable) como el
  **pronóstico final** se generan de forma recursiva, para que el MAPE reportado refleje de verdad
  el método que se está evaluando.


In [ ]:
# pip install "timesfm[torch]"   # ejecutar una sola vez en el entorno

import os
os.environ['USE_TF']    = '0'
os.environ['USE_TORCH'] = '1'

import ssl
ssl._create_default_https_context = ssl._create_unverified_context  # proxy corporativo Netskope (rutas stdlib / requests)

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

import requests
_orig_request = requests.Session.request
def _patched_request(self, *a, **kw):
    kw.setdefault('verify', False)
    return _orig_request(self, *a, **kw)
requests.Session.request = _patched_request  # huggingface_hub < 1.0 usa requests por debajo

# huggingface_hub >= 1.0 usa httpx en vez de requests -- el parche de arriba no le afecta.
try:
    import httpx

    def _unverified_httpx_client_factory():
        return httpx.Client(verify=False, follow_redirects=True, timeout=None)

    import huggingface_hub
    huggingface_hub.set_client_factory(_unverified_httpx_client_factory)
    print('huggingface_hub (backend httpx) configurado sin verificación SSL -- proxy Netskope')
except (ImportError, AttributeError):
    pass  # huggingface_hub < 1.0 (sin httpx o sin set_client_factory) -- ya cubierto por el parche de requests de arriba

import copy
import json
import warnings
import datetime as _dt

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import torch
import torch.nn.functional as F
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error

import timesfm
from timesfm.torch import util as tfm_util

warnings.filterwarnings('ignore')

print('Imports OK | torch:', torch.__version__, '| timesfm:', timesfm.__package__ and getattr(timesfm, '__version__', 'n/d'))


## 1. Configuración

In [ ]:
# MÉTRICA — cambiar aquí para alternar entre Parados / Afiliados / Demandantes (grupo ABC)
metrica = 'Parados'

# Nombre de fichero según la convención del proyecto (ver CLAUDE.md):
# "{Métrica} desde {AñoInicio} estatal.csv" — colocar el CSV junto a este notebook o ajustar la ruta.
CSV_PATH = f'{metrica} desde 2012 estatal.csv'

# Repo de Hugging Face del checkpoint TimesFM 2.5 (200M, Apache-2.0)
TIMESFM_REPO_ID = 'google/timesfm-2.5-200m-pytorch'

# Log-transform: solo para Contratos (grupo DE) -- sus oscilaciones son mucho
# más extremas en términos relativos (picos ~3-4x los valles) que Parados/
# Afiliados, donde quitamos el logaritmo porque no aportaba nada. En escala
# log esas oscilaciones se comprimen y se vuelven más simétricas, lo que
# debería ayudar a que la mediana (nuestro "punto") no se quede corta en los
# valles más profundos.
LOG_TRANSFORM = (metrica == 'Contratos')

VAL_MONTHS = 36   # meses reservados para el backtest (MAPE) -- igual que en el notebook principal

# --- FINE-TUNING (sin CV: un único entrenamiento, con los HP ya conocidos) --
# TimesFM 2.5 reshapea el contexto en parches de longitud fija (`model.p = 32`);
# `build_ft_windows` rellena automáticamente con ceros hasta el siguiente
# múltiplo de 32, así que FT_CTX puede ser cualquier valor.
# FT_CTX ya NO es un valor fijo -- se elige automáticamente por serie (ver
# sección 5): se prueba cada candidato de FT_CTX_GRID en el backtest y se usa
# el que dé mejor MAPE, tanto para el backtest como para el modelo final.
# Candidatos informados por el barrido manual hecho a mano para Parados
# (ganó 24) y Afiliados (ganó 36) -- cada métrica puede elegir uno distinto.
FT_CTX_GRID = [24, 36]   # los dos ganadores del barrido manual (Parados y Afiliados)
FT_HOR    = 12   # meses objetivo por ventana -- igual a RECURSIVE_STEP, para que el
                  # fine-tuning entrene exactamente el tamaño de paso que luego usa la recursión
FT_STEP   = 3    # paso entre ventanas de entrenamiento
FT_EPOCHS = 15   # épocas fijas, sin early stopping (igual que el notebook principal)

# Hiperparámetros FIJOS -- elegidos tras barrido manual (layers en [4,6,8],
# lr en [1e-6,5e-6,1e-5,2e-5,5e-5]) sobre la config final de contexto/paso
# recursivo (ctx=24/hor=12/step=12): layers=4,lr=5e-6 dio el mejor equilibrio
# de MAPE (3.29%), tendencia y estacionalidad de todo el barrido.
FIXED_LAYERS = 4       # nº de capas finales del transformer que se descongelan
FIXED_LR     = 5e-6    # learning rate del optimizador (AdamW)

# --- PRONÓSTICO RECURSIVO ----------------------------------------------------
# En vez de una única llamada a horizon=HORIZONTE_MESES, se encadenan pasos de
# RECURSIVE_STEP meses, añadiendo lo predicho en cada paso al contexto del
# siguiente -- igual que decode() encadena bloques de 128 cuando el horizonte
# pedido supera output_patch_len, pero aquí con bloques de 3.
# POINT_CHANNEL: qué canal de los 10 de cuantil se usa como "predicción
# puntual" -- 0 = media, 5 = mediana (comportamiento original de la API
# pública de timesfm). Se usa igual en fine-tuning (timesfm_forward_point)
# y en inferencia (recursive_forecast), para que no haya desajuste entre lo
# que se entrena y lo que se lee al predecir.
POINT_CHANNEL = 0

RECURSIVE_STEP = 12

SEED = 11   # misma semilla que set_random_seed(11) en el script NP

print(f'Métrica: {metrica}')
print(f'Fine-tuning FIJO: layers={FIXED_LAYERS}  lr={FIXED_LR}  (sin CV)')
print(f'Pronóstico recursivo: bloques de {RECURSIVE_STEP} meses')


## 2. Carga de datos y preparación

In [ ]:
# Misma convención de carga que forecast_ABC_estatal_NP.py
df = pd.read_csv(CSV_PATH, sep=';', na_values=["'-"])
df['Fecha'] = pd.to_datetime(df['Fecha'], dayfirst=True)
df[metrica] = pd.to_numeric(df[metrica], errors='coerce')
df = df.sort_values('Fecha').reset_index(drop=True)

# all_values_model es lo que ve el modelo: la serie tal cual, o su logaritmo
# si LOG_TRANSFORM=True (ver celda de configuración). to_raw() deshace esa
# transformación antes de comparar/mostrar/exportar nada.
all_values       = df[metrica].values.astype(np.float32)
all_values_model = np.log(all_values) if LOG_TRANSFORM else all_values
n = len(all_values)


def to_raw(x):
    """Deshace la transformación del modelo (log, si LOG_TRANSFORM=True) -- escala real."""
    return np.exp(x) if LOG_TRANSFORM else x


# --- HORIZONTE DE PRONÓSTICO DINÁMICO (igual que forecast_ABC_estatal_NP.py) ---
now = _dt.datetime.now().year
f_end = f'{now + 3}-12'
FECHA_FIN_PRONOSTICO = pd.Timestamp(f_end + '-01')
UltimaFechaHistorico  = df['Fecha'].iloc[-1]
HORIZONTE_MESES = (
    (FECHA_FIN_PRONOSTICO.year  - UltimaFechaHistorico.year)  * 12 +
    (FECHA_FIN_PRONOSTICO.month - UltimaFechaHistorico.month)
)

print(f'Total: {n} meses | Rango: {df["Fecha"].iloc[0]:%Y-%m} — {df["Fecha"].iloc[-1]:%Y-%m}')
print(f"Último dato: {UltimaFechaHistorico.strftime('%Y-%m')}. Horizonte de pronóstico: {HORIZONTE_MESES} meses (hasta {f_end}).")
print(f'LOG_TRANSFORM: {LOG_TRANSFORM}')

if n < FT_CTX_GRID[0] + VAL_MONTHS:
    raise ValueError(f'Histórico insuficiente ({n}m) para contexto={FT_CTX_GRID[0]}m + validación={VAL_MONTHS}m')


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df['Fecha'], df[metrica])
plt.title(f'{metrica} — histórico')
plt.xlabel('Fecha')
plt.ylabel(metrica)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 3. TimesFM 2.5 — descarga y carga del modelo base

Dos formas de conseguir el checkpoint (`config.json` + `model.safetensors`, ~800MB): automática
(`from_pretrained`, puede fallar con `403` detrás de un proxy corporativo) o manual (descargar desde
el navegador y apuntar `TIMESFM_LOCAL_DIR` a la carpeta local). Ver el notebook principal para más
detalle -- aquí solo el código, ya probado.

In [ ]:
# Ruta local del checkpoint. Ajusta si lo guardas en otro sitio.
TIMESFM_LOCAL_DIR = r'C:\Users\sgei044\Desktop\ML and IA with Python\Parados Contratos Afiliados 2026-2028\Modelos\TimesFM\timesfm-2.5-200m-pytorch'

if os.path.exists(os.path.join(TIMESFM_LOCAL_DIR, 'model.safetensors')):
    print(f'Cargando checkpoint local desde: {TIMESFM_LOCAL_DIR}')
    model = timesfm.TimesFM_2p5_200M_torch(torch_compile=False)
    model.load_checkpoint(TIMESFM_LOCAL_DIR)
else:
    print('No se encontró checkpoint local -- intentando descarga automática desde Hugging Face...')
    model = timesfm.TimesFM_2p5_200M_torch.from_pretrained(TIMESFM_REPO_ID, torch_compile=False)

model.compile(
    timesfm.ForecastConfig(
        max_context=1024,   # de sobra para toda la serie (histórico completo + lo ya predicho)
        max_horizon=128,    # >> RECURSIVE_STEP, múltiplo de 128 (output_patch_len)
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        # False: por defecto este flag promedia la predicción con la que da el
        # modelo sobre la serie NEGADA (-serie) -- no tiene sentido para datos
        # económicos siempre positivos (ver notebook principal para el detalle).
        force_flip_invariance=False,
        infer_is_positive=True,
        fix_quantile_crossing=True,
    )
)

base_module = copy.deepcopy(model.model)   # copia intacta del modelo preentrenado

# Constantes de arquitectura (verificadas contra el código fuente de timesfm==3.0.0)
TFM_P          = model.model.p            # 32   — longitud de parche de entrada
TFM_O          = model.model.o            # 128  — longitud de parche de salida
TFM_Q          = model.model.q            # 10   — canales de cuantil (0=media, 1..9=deciles 0.1..0.9)
TFM_ARIDX      = model.model.aridx        # 5    — canal usado como estimación puntual (mediana, p50)
TFM_NUM_LAYERS = len(model.model.stacked_xf)  # 20 — capas de transformer

Q_LOW, Q_MED, Q_HIGH = 1, TFM_ARIDX, 9    # p10, p50 (mediana=punto), p90

print(f'Modelo cargado — parámetros: {sum(p.numel() for p in model.model.parameters()):,}')
print(f'p={TFM_P}  o={TFM_O}  q={TFM_Q}  aridx={TFM_ARIDX}  num_layers={TFM_NUM_LAYERS}')


## 4. Funciones auxiliares

`timesfm_forward_point` / `pad_context_to_patch` / `build_ft_windows` / `make_ft_module` /
`run_ft_training` son idénticas al notebook principal (ya verificadas contra el código fuente real
de `timesfm==3.0.0` -- ver ese notebook para el detalle de la verificación).

`recursive_forecast` es la única función nueva de este notebook: encadena llamadas a la API pública
`model.forecast()` en bloques de `RECURSIVE_STEP` meses, añadiendo lo predicho en cada paso al
contexto del siguiente.

In [ ]:
def timesfm_forward_point(core_module, context_batch, mask_batch, horizon):
    """Pronóstico puntual diferenciable para fine-tuning.

    context_batch: FloatTensor (B, ctx_len); ctx_len debe ser múltiplo de core_module.p.
    mask_batch: BoolTensor (B, ctx_len); True donde context_batch es relleno (no datos reales).
    horizon: int <= core_module.o (128).
    Devuelve: FloatTensor (B, horizon).
    """
    p, o = core_module.p, core_module.o
    B, ctx_len = context_batch.shape
    assert ctx_len % p == 0, f'ctx_len ({ctx_len}) debe ser múltiplo de {p}'
    assert horizon <= o, f'horizon ({horizon}) debe ser <= {o}'

    patched_inputs = context_batch.reshape(B, -1, p)
    patched_masks  = mask_batch.reshape(B, -1, p)

    n_pts = torch.zeros(B, device=context_batch.device)
    mu    = torch.zeros(B, device=context_batch.device)
    sigma = torch.zeros(B, device=context_batch.device)
    patch_mu, patch_sigma = [], []
    for i in range(patched_inputs.shape[1]):
        (n_pts, mu, sigma), _ = tfm_util.update_running_stats(
            n_pts, mu, sigma, patched_inputs[:, i], patched_masks[:, i]
        )
        patch_mu.append(mu)
        patch_sigma.append(sigma)
    context_mu    = torch.stack(patch_mu, dim=1)
    context_sigma = torch.stack(patch_sigma, dim=1)

    normed_inputs = tfm_util.revin(patched_inputs, context_mu, context_sigma, reverse=False)
    normed_inputs = torch.where(patched_masks, 0.0, normed_inputs)

    (_, _, normed_outputs, _), _ = core_module(normed_inputs, patched_masks, decode_caches=None)

    renormed_outputs = tfm_util.revin(normed_outputs, context_mu, context_sigma, reverse=True)
    renormed_outputs = renormed_outputs.reshape(B, -1, o, core_module.q)

    # POINT_CHANNEL (0 = media, 5 = mediana/aridx) -- se entrena contra el
    # MISMO canal que luego se usa en recursive_forecast, para que no haya
    # desajuste entre lo que se optimiza y lo que se lee al predecir.
    return renormed_outputs[:, -1, :horizon, POINT_CHANNEL]


def pad_context_to_patch(ctx_arr, patch_len=32):
    """Rellena ctx_arr con ceros al PRINCIPIO hasta el siguiente múltiplo de
    patch_len (los datos reales quedan siempre al final). Devuelve (ctx_padded,
    mask) -- mask=True marca las posiciones de relleno."""
    L = len(ctx_arr)
    padded_len = ((L + patch_len - 1) // patch_len) * patch_len
    pad_amount = padded_len - L
    ctx_padded = np.concatenate([np.zeros(pad_amount, dtype=ctx_arr.dtype), ctx_arr])
    mask = np.concatenate([np.ones(pad_amount, dtype=bool), np.zeros(L, dtype=bool)])
    return ctx_padded, mask


def build_ft_windows(series, ctx, hor, step, patch_len=32):
    """Ternas (contexto rellenado, máscara, objetivo) por ventana deslizante."""
    windows = []
    for i in range(ctx, len(series) - hor + 1, step):
        ctx_arr, mask = pad_context_to_patch(series[i - ctx: i], patch_len)
        tgt_arr = series[i: i + hor]
        windows.append((ctx_arr, mask, tgt_arr))
    return windows


def make_ft_module(n_layers, base=None):
    """Copia profunda de `base` con solo las últimas `n_layers` capas de transformer
    + la cabeza de proyección puntual descongeladas."""
    base = base if base is not None else base_module
    ft_m = copy.deepcopy(base)
    for param in ft_m.parameters():
        param.requires_grad = False
    total = len(ft_m.stacked_xf)
    for i in range(total - n_layers, total):
        for param in ft_m.stacked_xf[i].parameters():
            param.requires_grad = True
    for param in ft_m.output_projection_point.parameters():
        param.requires_grad = True
    return ft_m


def run_ft_training(ft_m, lr, epochs, windows, seed=SEED, verbose=False):
    """Fine-tuning con AdamW, épocas FIJAS (sin early stopping)."""
    trainable_p = [p for p in ft_m.parameters() if p.requires_grad]
    optimizer   = torch.optim.AdamW(trainable_p, lr=lr)
    torch.manual_seed(seed)
    ft_m.train()
    for epoch in range(1, epochs + 1):
        epoch_losses = []
        for ctx_arr, mask_arr, tgt_arr in windows:
            x = torch.tensor(ctx_arr, dtype=torch.float32).unsqueeze(0)
            m = torch.tensor(mask_arr, dtype=torch.bool).unsqueeze(0)
            y = torch.tensor(tgt_arr, dtype=torch.float32).unsqueeze(0)
            pred = timesfm_forward_point(ft_m, x, m, horizon=len(tgt_arr))
            loss = F.mse_loss(pred, y)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_p, 1.0)
            optimizer.step()
            epoch_losses.append(loss.item())
        if verbose:
            print(f'    Época {epoch}/{epochs}: loss medio = {np.mean(epoch_losses):.6f}')
    ft_m.eval()
    return ft_m


def recursive_forecast(compiled_model, module, context, total_horizon, step=RECURSIVE_STEP):
    """Pronóstico de `total_horizon` meses encadenando pasos de `step` meses:
    en cada paso predice `step` meses con todo el contexto disponible hasta
    ese momento (histórico real + lo ya predicho en pasos anteriores), y añade
    lo predicho al contexto del siguiente paso -- igual que decode() encadena
    bloques de 128 cuando el horizonte pedido supera output_patch_len, pero
    aquí con bloques de `step`.

    compiled_model: el objeto timesfm.TimesFM_2p5_200M_torch ya compilado.
    module: qué pesos usar (base_module para zero-shot, ft_model para fine-tuned).
    context: array 1D con el histórico real de partida.
    Devuelve: (point_forecast, quantile_forecast). point_forecast usa
    POINT_CHANNEL (0=media por defecto), NO el canal 5 (mediana) que devuelve
    `compiled_model.forecast()` por defecto -- se ignora ese primer valor y se
    lee directamente del array de cuantiles completo, para poder elegir canal.
    """
    compiled_model.model = module
    ctx = list(context)
    points, quants = [], []
    remaining = total_horizon
    while remaining > 0:
        h = min(step, remaining)
        _, quant_block = compiled_model.forecast(horizon=h, inputs=[np.array(ctx)])
        quant_block = quant_block[0]              # (h, 10)
        point_block = quant_block[:, POINT_CHANNEL]  # (h,)
        points.append(point_block)
        quants.append(quant_block)
        ctx = ctx + list(point_block)  # lo predicho pasa a formar parte del contexto
        remaining -= h
    return np.concatenate(points), np.concatenate(quants, axis=0)


print('Funciones auxiliares definidas.')


## 5. Selección automática de `FT_CTX` + Fine-tuning para el BACKTEST

En vez de fijar `FT_CTX` a mano, se prueba cada candidato de `FT_CTX_GRID` con un fine-tuning
completo (siempre sin los últimos `VAL_MONTHS` meses, igual que antes, para que el MAPE del
backtest siga siendo honesto) y se elige el que dé mejor MAPE recursivo. El ganador se guarda en
`FT_CTX` y se reutiliza en la sección 7 para el fine-tuning FINAL (100% del histórico) -- así cada
métrica (Parados, Afiliados, ...) puede acabar con un `FT_CTX` distinto sin tocar nada a mano.

In [ ]:
print(f'Buscando el mejor FT_CTX entre {FT_CTX_GRID} (layers={FIXED_LAYERS}, lr={FIXED_LR} fijos)...')

backtest_context = all_values_model[:n - VAL_MONTHS]
backtest_actual  = all_values[n - VAL_MONTHS:]

best_ctx_mape      = float('inf')
ctx_grid_results   = []
ft_module_backtest = None
ft_bt_point        = None
ft_bt_quant        = None

for candidate_ctx in FT_CTX_GRID:
    ft_windows_bt = build_ft_windows(all_values_model[:-VAL_MONTHS], candidate_ctx, FT_HOR, FT_STEP)
    ft_m_bt = make_ft_module(FIXED_LAYERS)
    ft_m_bt = run_ft_training(ft_m_bt, FIXED_LR, FT_EPOCHS, ft_windows_bt)

    bt_point_model, bt_quant_model = recursive_forecast(model, ft_m_bt, backtest_context, VAL_MONTHS)
    bt_point = to_raw(bt_point_model)
    mape_ctx = mean_absolute_percentage_error(backtest_actual, bt_point) * 100
    ctx_grid_results.append({'FT_CTX': candidate_ctx, 'mape': round(mape_ctx, 2)})
    print(f'  FT_CTX={candidate_ctx}: MAPE={mape_ctx:.2f}%')

    if mape_ctx < best_ctx_mape:
        best_ctx_mape       = mape_ctx
        FT_CTX              = candidate_ctx
        ft_module_backtest  = ft_m_bt
        ft_bt_point         = bt_point
        ft_bt_quant         = to_raw(bt_quant_model)
    else:
        del ft_m_bt

mape_fine_tuned = best_ctx_mape

df_ctx_grid = pd.DataFrame(ctx_grid_results).sort_values('mape').reset_index(drop=True)
display(df_ctx_grid)
print(f'\nMejor FT_CTX para {metrica}: {FT_CTX}  (MAPE {mape_fine_tuned:.2f}%)')


## 6. Backtest recursivo (últimos {VAL_MONTHS} meses) — Zero-Shot vs Fine-tuned

Mismo procedimiento (recursivo, bloques de `RECURSIVE_STEP` meses) para los dos modelos, así el MAPE
es comparable entre sí.

In [ ]:
zs_bt_point_model, zs_bt_quant_model = recursive_forecast(model, base_module, backtest_context, VAL_MONTHS)
zs_bt_point = to_raw(zs_bt_point_model)
zs_bt_quant = to_raw(zs_bt_quant_model)
mape_zero_shot = mean_absolute_percentage_error(backtest_actual, zs_bt_point) * 100
print(f'Zero-Shot recursivo (últimos {VAL_MONTHS}m) — MAPE: {mape_zero_shot:.2f}%')
print(f'Fine-tuned recursivo (últimos {VAL_MONTHS}m) — MAPE: {mape_fine_tuned:.2f}%  (FT_CTX={FT_CTX})')


In [ ]:
val_dates = df['Fecha'].iloc[-VAL_MONTHS:].reset_index(drop=True)

# un poco de histórico real antes de la validación, para dar contexto visual
context_months_plot = 24
plot_from = max(0, n - VAL_MONTHS - context_months_plot)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['Fecha'].iloc[plot_from:], df[metrica].iloc[plot_from:], color='steelblue', linewidth=2, label='Histórico real')

ax.plot(val_dates, ft_bt_point, color='crimson', linestyle='--', linewidth=2.5,
        label=f'Fine-tuned recursivo (MAPE {mape_fine_tuned:.2f}%)')
ax.plot(val_dates, zs_bt_point, color='gray', linestyle=':', linewidth=2,
        label=f'Zero-Shot recursivo (MAPE {mape_zero_shot:.2f}%)')

ax.axvline(x=val_dates.iloc[0], color='green', linestyle=':', alpha=0.8, label='Inicio validación')
ax.set_title(f'{metrica} — Backtest recursivo (últimos {VAL_MONTHS} meses, bloques de {RECURSIVE_STEP}m)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel(metrica)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 7. Fine-tuning FINAL (100% del histórico)

Ahora sí, con todos los datos -- este es el modelo (`ft_model_final`) que genera el pronóstico real
de la sección siguiente. Es un entrenamiento distinto del de la sección 5 (ese solo era para poder
medir el MAPE de forma honesta).

In [ ]:
print(f'Fine-tuning FINAL: layers={FIXED_LAYERS}, lr={FIXED_LR}, sobre el 100% del histórico...')

ft_windows_final = build_ft_windows(all_values_model, FT_CTX, FT_HOR, FT_STEP)
print(f'Ventanas de entrenamiento: {len(ft_windows_final)} (contexto={FT_CTX}m, objetivo={FT_HOR}m)')

ft_model_final = make_ft_module(FIXED_LAYERS)
ft_model_final = run_ft_training(ft_model_final, FIXED_LR, FT_EPOCHS, ft_windows_final, verbose=True)

print('Fine-tuning final terminado.')


## 8. Pronóstico final recursivo (Zero-Shot y Fine-tuned)

In [ ]:
forecast_dates = pd.date_range(
    UltimaFechaHistorico + pd.DateOffset(months=1),
    periods=HORIZONTE_MESES, freq='MS'
)

print(f'Generando pronóstico recursivo hasta {f_end} ({HORIZONTE_MESES} meses, en bloques de {RECURSIVE_STEP})...')

zs_forecast_point_model, zs_forecast_quant_model = recursive_forecast(model, base_module, all_values_model, HORIZONTE_MESES)
zs_forecast_point = to_raw(zs_forecast_point_model)
zs_forecast_quant = to_raw(zs_forecast_quant_model)

forecast_point_model, forecast_quant_model = recursive_forecast(model, ft_model_final, all_values_model, HORIZONTE_MESES)
forecast_point = to_raw(forecast_point_model)
forecast_quant = to_raw(forecast_quant_model)

print(f'Pronóstico: {forecast_dates[0]:%Y-%m} -> {forecast_dates[-1]:%Y-%m}')
display(pd.DataFrame({
    f'{metrica}_FineTuned': forecast_point.round(0),
    f'{metrica}_ZeroShot':  zs_forecast_point.round(0),
}, index=forecast_dates).head(6))


## 9. Resultado (formato del proyecto) y exportación a Excel

In [ ]:
historico = [
    {'fecha': row['Fecha'].strftime('%Y-%m'), 'valor': round(float(row[metrica])) if pd.notna(row[metrica]) else None}
    for _, row in df.iterrows()
]
pronostico = [
    {'fecha': d.strftime('%Y-%m'), 'valor': round(float(v))}
    for d, v in zip(forecast_dates, forecast_point)
]
intervalo_confianza = {
    'superior': [{'fecha': d.strftime('%Y-%m'), 'valor': round(float(v))}
                 for d, v in zip(forecast_dates, forecast_quant[:, Q_HIGH])],
    'inferior': [{'fecha': d.strftime('%Y-%m'), 'valor': round(float(v))}
                 for d, v in zip(forecast_dates, forecast_quant[:, Q_LOW])],
}

result = {
    'metrica':             metrica,
    'modo':                'estatal',
    'modelo':              'TimesFM-recursivo',
    'atributo':            None,
    'anio_inicio':         int(df['Fecha'].iloc[0].year),
    'historico':           historico,
    'pronostico':          pronostico,
    'intervalo_confianza': intervalo_confianza,
    'mape':                round(float(mape_fine_tuned), 2),
    'hiperparametros':     {'layers': FIXED_LAYERS, 'lr': FIXED_LR, 'recursive_step': RECURSIVE_STEP},
    # Extra fuera del contrato del proyecto -- solo para comparar en el notebook.
    'zero_shot_referencia': {
        'mape_val_holdout': round(float(mape_zero_shot), 2),
        'pronostico': [
            {'fecha': d.strftime('%Y-%m'), 'valor': round(float(v))}
            for d, v in zip(forecast_dates, zs_forecast_point)
        ],
    },
}

df_fc = pd.DataFrame({
    'Fecha': forecast_dates,
    f'{metrica}_FineTuned': forecast_point.round(0),
    'p10_FineTuned':        forecast_quant[:, Q_LOW].round(0),
    'p90_FineTuned':        forecast_quant[:, Q_HIGH].round(0),
    f'{metrica}_ZeroShot':  zs_forecast_point.round(0),
    'p10_ZeroShot':         zs_forecast_quant[:, Q_LOW].round(0),
    'p90_ZeroShot':         zs_forecast_quant[:, Q_HIGH].round(0),
})

out_file = f'MAPE {metrica} {f_end} TimesFM-recursivo.xlsx'
with pd.ExcelWriter(out_file) as w:
    df_fc.to_excel(w, sheet_name='Pronostico', index=False)
print(f'Exportado: {out_file}')


## 10. Visualización del pronóstico final

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df['Fecha'], df[metrica], color='steelblue', linewidth=2, label='Histórico')

ax.plot(forecast_dates, forecast_point, color='crimson', linestyle='--', linewidth=2.5,
        label=f'TimesFM fine-tuned recursivo (MAPE {mape_fine_tuned:.2f}%)')
ax.fill_between(forecast_dates, forecast_quant[:, Q_LOW], forecast_quant[:, Q_HIGH],
                color='crimson', alpha=0.15, label='Intervalo p10-p90 (fine-tuned)')

ax.plot(forecast_dates, zs_forecast_point, color='gray', linestyle=':', linewidth=2,
        label=f'TimesFM Zero-Shot recursivo (MAPE {mape_zero_shot:.2f}%)')
ax.fill_between(forecast_dates, zs_forecast_quant[:, Q_LOW], zs_forecast_quant[:, Q_HIGH],
                color='gray', alpha=0.10, label='Intervalo p10-p90 (Zero-Shot)')

ax.axvline(x=forecast_dates[0], color='green', linestyle=':', alpha=0.8, label='Inicio pronóstico')
ax.set_title(f'{metrica} — TimesFM 2.5 recursivo (bloques de {RECURSIVE_STEP}m) — Pronóstico hasta {f_end}',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel(metrica)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.tick_params(axis='x', rotation=45)
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 11. Resumen

In [ ]:
print('=' * 70)
print(f'  TimesFM 2.5 RECURSIVO — {metrica} | bloques de {RECURSIVE_STEP} meses | layers={FIXED_LAYERS} lr={FIXED_LR}')
print('=' * 70)
print(f'  Zero-Shot recursivo  — MAPE ({VAL_MONTHS}m): {mape_zero_shot:.2f}%')
print(f'  Fine-tuned recursivo — MAPE ({VAL_MONTHS}m): {mape_fine_tuned:.2f}%')
print('-' * 70)
print(f'  Pronóstico FINE-TUNED {forecast_dates[0]:%Y-%m} - {forecast_dates[-1]:%Y-%m}:')
print(f'    inicio {forecast_point[0]:>10,.0f}   fin {forecast_point[-1]:>10,.0f}')
print(f'    p10    inicio {forecast_quant[0, Q_LOW]:>10,.0f}   fin {forecast_quant[-1, Q_LOW]:>10,.0f}')
print(f'    p90    inicio {forecast_quant[0, Q_HIGH]:>10,.0f}   fin {forecast_quant[-1, Q_HIGH]:>10,.0f}')
print('-' * 70)
print(f'  Pronóstico ZERO-SHOT {forecast_dates[0]:%Y-%m} - {forecast_dates[-1]:%Y-%m}:')
print(f'    inicio {zs_forecast_point[0]:>10,.0f}   fin {zs_forecast_point[-1]:>10,.0f}')
print(f'    p10    inicio {zs_forecast_quant[0, Q_LOW]:>10,.0f}   fin {zs_forecast_quant[-1, Q_LOW]:>10,.0f}')
print(f'    p90    inicio {zs_forecast_quant[0, Q_HIGH]:>10,.0f}   fin {zs_forecast_quant[-1, Q_HIGH]:>10,.0f}')
print('=' * 70)
